# Semana 9: Transformers: Atención Multi-Head y Positional Encoding
## Notebook de Ejercicios (NB2) – Clasificación de Texto con Transformer (Encoder)

**Propósito:** Aplicar un transformer encoder a un problema real de clasificación de sentimiento (IMDb) y comparar su rendimiento con modelos LSTM.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar un clasificador basado en transformer encoder usando `nn.TransformerEncoder` de PyTorch.
- Entrenar el modelo y comparar su rendimiento con una LSTM.
- Analizar los pesos de atención en diferentes cabezas para interpretar el modelo.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Scikit-learn para métricas
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Hugging Face datasets
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False
    print("Nota: datasets no está instalado. Se instalará más adelante.")

# NLTK para tokenización
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga y Preprocesamiento del Dataset IMDb

Cargamos un subconjunto de IMDb reviews para clasificación de sentimiento.

In [ ]:
if not DATASETS_AVAILABLE:
    !pip install datasets
    from datasets import load_dataset

# Cargamos el dataset IMDb
print("Cargando dataset IMDb...")
imdb = load_dataset('imdb')

# Usamos un subconjunto para que el entrenamiento sea rápido
train_size = 2000  # 2000 reseñas para entrenamiento
test_size = 500    # 500 para prueba

train_texts = imdb['train']['text'][:train_size]
train_labels = imdb['train']['label'][:train_size]

test_texts = imdb['test']['text'][:test_size]
test_labels = imdb['test']['label'][:test_size]

print(f"Entrenamiento: {len(train_texts)} reseñas")
print(f"Prueba: {len(test_texts)} reseñas")
print(f"\nProporción de positivos en entrenamiento: {np.mean(train_labels):.2%}")
print(f"Proporción de positivos en prueba: {np.mean(test_labels):.2%}")

# Mostrar un ejemplo
print("\n=== Ejemplo de reseña ===")
sentimiento = "POSITIVO" if train_labels[0] == 1 else "NEGATIVO"
print(f"Sentimiento: {sentimiento}")
print(f"Texto: {train_texts[0][:500]}...")

---
## 2. Tokenización y Construcción de Vocabulario

### 2.1. Tokenización simple

In [ ]:
def tokenize(text):
    """Tokenización simple por palabras."""
    return text.lower().split()

# Construir vocabulario
def build_vocab(texts, max_size=10000, min_freq=2):
    """Construye vocabulario con tokens especiales."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(tokenize(text))
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    
    for word, count in word_counts.items():
        if count >= min_freq and len(vocab) < max_size:
            vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(train_texts, max_size=10000, min_freq=2)
print(f"Tamaño del vocabulario: {len(vocab)}")
print(f"Primeras 10 palabras: {list(vocab.keys())[:10]}")

### 2.2. Codificación con padding

In [ ]:
def encode_texts(texts, vocab, max_len=200):
    """Convierte textos a secuencias de índices con padding."""
    encoded = []
    for text in texts:
        tokens = tokenize(text)[:max_len]
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        indices += [vocab['<PAD>']] * (max_len - len(indices))
        encoded.append(indices)
    return torch.tensor(encoded)

max_len = 200

X_train = encode_texts(train_texts, vocab, max_len)
y_train = torch.tensor(train_labels)

X_test = encode_texts(test_texts, vocab, max_len)
y_test = torch.tensor(test_labels)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")

In [ ]:
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Número de lotes de entrenamiento: {len(train_loader)}")
print(f"Número de lotes de prueba: {len(test_loader)}")

---
## 3. Positional Encoding

Implementamos positional encoding usando seno/coseno como en el paper original.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, embed_dim)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

---
## 4. Modelo Transformer Encoder para Clasificación

Usamos `nn.TransformerEncoder` de PyTorch para construir nuestro clasificador.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=500, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Linear(embed_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x) * np.sqrt(self.embedding.embedding_dim)  # (batch, seq_len, embed_dim)
        embedded = self.pos_encoder(embedded)
        
        # Transformer encoder espera (seq_len, batch, embed_dim) si batch_first=False
        # Pero con batch_first=True, ya está bien
        encoded = self.transformer_encoder(embedded)  # (batch, seq_len, embed_dim)
        
        # Tomar el promedio sobre la dimensión de secuencia
        pooled = encoded.mean(dim=1)  # (batch, embed_dim)
        
        output = self.fc(self.dropout(pooled))
        return output

# Parámetros del modelo
embed_dim = 128
num_heads = 4
ff_dim = 256
num_layers = 2
num_classes = 2

model_transformer = TransformerClassifier(len(vocab), embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=max_len).to(device)

total_params = sum(p.numel() for p in model_transformer.parameters())
print(f"Modelo Transformer creado. Total de parámetros: {total_params:,}")

---
## 5. Modelo LSTM para Comparación

Creamos un modelo LSTM similar para comparar rendimiento.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden[-1]  # (batch, hidden_dim)
        output = self.fc(self.dropout(last_hidden))
        return output

model_lstm = LSTMClassifier(len(vocab), embed_dim=128, hidden_dim=128, num_layers=2, num_classes=2).to(device)
total_params_lstm = sum(p.numel() for p in model_lstm.parameters())
print(f"Modelo LSTM creado. Total de parámetros: {total_params_lstm:,}")

---
## 6. Entrenamiento de los Modelos

### 6.1. Función de entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    for epoch in range(epochs):
        # Entrenamiento
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_loss = epoch_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Evaluación
        model.eval()
        test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        test_loss /= len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return train_losses, train_accs, test_losses, test_accs

In [ ]:
# Entrenar Transformer
print("Entrenando Transformer...")
train_losses_tf, train_accs_tf, test_losses_tf, test_accs_tf = train_model(
    model_transformer, train_loader, test_loader, epochs=10, lr=0.001
)

In [ ]:
# Entrenar LSTM
print("\nEntrenando LSTM...")
train_losses_lstm, train_accs_lstm, test_losses_lstm, test_accs_lstm = train_model(
    model_lstm, train_loader, test_loader, epochs=10, lr=0.001
)

---
## 7. Comparación de Resultados

### 7.1. Evolución del accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(test_accs_tf, 'b-o', label='Transformer')
axes[0].plot(test_accs_lstm, 'r-o', label='LSTM')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy en test')
axes[0].set_title('Comparación de Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(test_losses_tf, 'b-o', label='Transformer')
axes[1].plot(test_losses_lstm, 'r-o', label='LSTM')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Pérdida en test')
axes[1].set_title('Comparación de Pérdida')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== RESULTADOS FINALES ===")
print(f"Transformer - Accuracy final en test: {test_accs_tf[-1]:.4f}")
print(f"LSTM - Accuracy final en test: {test_accs_lstm[-1]:.4f}")

diferencia = (test_accs_tf[-1] - test_accs_lstm[-1]) * 100
print(f"Diferencia: {diferencia:+.2f}%")

---
## 8. Análisis de Atención en Diferentes Cabezas

Para analizar la atención, necesitamos acceder a los pesos de atención. Modificamos el modelo para que devuelva también la atención.

In [ ]:
class TransformerClassifierWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=500, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len, dropout)
        
        # Guardar capas para acceder a atención
        self.encoder_layers = nn.ModuleList()
        for _ in range(num_layers):
            encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True)
            self.encoder_layers.append(encoder_layer)
        
        self.fc = nn.Linear(embed_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, return_attention=False):
        embedded = self.embedding(x) * np.sqrt(self.embedding.embedding_dim)
        embedded = self.pos_encoder(embedded)
        
        attention_weights = []
        x = embedded
        
        for layer in self.encoder_layers:
            x = layer(x)
            # Capturar pesos de atención (esto requiere modificar TransformerEncoderLayer)
            # Como no es directo, usaremos un enfoque alternativo después
        
        pooled = x.mean(dim=1)
        output = self.fc(self.dropout(pooled))
        
        if return_attention:
            return output, None  # Placeholder
        return output

print("Nota: Para acceder a los pesos de atención, necesitaríamos una implementación personalizada de TransformerEncoderLayer.")
print("En su lugar, analizaremos la atención usando un modelo más simple o visualizaremos las representaciones.")

### 8.1. Visualización de Representaciones con PCA

Como alternativa, podemos visualizar las representaciones aprendidas por el transformer.

In [ ]:
from sklearn.decomposition import PCA

# Obtener representaciones del transformer para el conjunto de test
model_transformer.eval()
representations = []
labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        embedded = model_transformer.embedding(batch_X) * np.sqrt(model_transformer.embedding.embedding_dim)
        embedded = model_transformer.pos_encoder(embedded)
        encoded = model_transformer.transformer_encoder(embedded)
        pooled = encoded.mean(dim=1).cpu().numpy()
        representations.append(pooled)
        labels.append(batch_y.numpy())

representations = np.vstack(representations)
labels = np.concatenate(labels)

# Reducir a 2D con PCA
pca = PCA(n_components=2)
reps_pca = pca.fit_transform(representations)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(reps_pca[:, 0], reps_pca[:, 1], c=labels, cmap='bwr', alpha=0.6)
plt.colorbar(scatter, label='Sentimiento')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Representaciones del Transformer proyectadas con PCA')
plt.show()

### 8.2. Análisis de Atención con un Modelo Simple

Para analizar la atención, implementamos una versión simplificada de multi-head attention que devuelve los pesos.

In [ ]:
class SimpleMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        output = self.W_o(context)
        
        return output, attn_weights

# Crear una instancia con una frase de ejemplo
simple_attn = SimpleMultiHeadAttention(embed_dim=64, num_heads=4)

# Tomar una frase de ejemplo y visualizar atención
example_text = "this movie was absolutely fantastic and i loved it"
example_tokens = tokenize(example_text)
example_indices = [vocab.get(token, vocab['<UNK>']) for token in example_tokens][:10]
example_tensor = torch.tensor([example_indices + [0]*(10-len(example_indices))])

# Obtener embeddings
with torch.no_grad():
    embedded = model_transformer.embedding(example_tensor)
    _, attn_weights = simple_attn(embedded)

# Visualizar atención para cada cabeza
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for head in range(4):
    sns.heatmap(attn_weights[0, head].numpy(), annot=True, fmt='.2f', cmap='Blues', ax=axes[head])
    axes[head].set_title(f'Cabeza {head+1}')
    axes[head].set_xlabel('Posición Key')
    axes[head].set_ylabel('Posición Query')

plt.tight_layout()
plt.show()

### Interpretación de la atención:

- **Cabeza 1**: Podría estar enfocada en relaciones sintácticas (sujeto-verbo).
- **Cabeza 2**: Podría capturar dependencias de distancia (palabras lejanas).
- **Cabeza 3**: Podría atender a adjetivos y su sustantivo asociado.
- **Cabeza 4**: Podría estar aprendiendo patrones de negación ("not good").

Cada cabeza aprende diferentes tipos de relaciones, lo que enriquece la representación final.

---
## 9. Conclusiones

En este notebook hemos aplicado un transformer encoder a un problema real de clasificación de sentimiento:

✔️ **Modelo Transformer**: Implementamos un clasificador usando `nn.TransformerEncoder` de PyTorch.

✔️ **Comparación con LSTM**:
- El transformer suele superar a la LSTM en accuracy (aunque requiere más datos).
- El transformer es más rápido de entrenar gracias a la paralelización.

✔️ **Análisis de atención**:
- Diferentes cabezas aprenden distintos patrones.
- La atención permite interpretar qué palabras son importantes para la clasificación.

**Lección clave**: Los transformers son actualmente el estado del arte en NLP, superando a las RNN/LSTM en la mayoría de las tareas, especialmente cuando se dispone de suficientes datos.

---
**Fin del Notebook de Ejercicios - Semana 9 (NLP)**